# 《PythAPCS123》單元 13-8：考場系統化除錯戰略：錯誤重現、二分隔離、斷言與送出前 SOP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-8_systematic_debugging_and_pre_submission_checklist.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：整合全章所有除錯與防禦內功，融會貫通為一套在緊迫考場高壓下依然能冷靜運轉的「系統化除錯戰略工作流」！徹底告別盲目亂改代碼越改越爛的恐慌，學會構建 3 行極小重現案例（MRE）、掌握分治二分註解法 10 秒內鎖定問題行段、熟練使用帶標籤的 Print 動態追蹤迴圈內部狀態、在關鍵演算法節點設下 `assert` 不變量防衛線、並隨身攜帶邊界測資（Corner Cases）自測清單。最後嚴格執行「送出前一分鐘黃金檢查 SOP」，為進入第十四章 19 題 APCS 真題大考驗做好 100% 的萬全準備！


### 13.8.1 系統化除錯第一步：建立極小可重現測試案例（Minimal Reproducible Example）

當你在 APCS 考場上寫完一段 60 行的演算法，按下一組包含 1000 個數字的大測資進行測試，結果螢幕上噴出了一個錯誤答案時，許多考生的第一反應是：「盯著那 1000 個數字和 60 行代碼發呆，腦袋一片空白」。

面對龐大混亂的資料與長篇代碼，人類的大腦是無法直接進行微觀分析的。全世界頂尖工程師與競賽選手的第一本能，永遠是：**「建立極小可重現測試案例（Minimal Reproducible Example, MRE）」**！

MRE 的核心法則：
1. **極小化資料規模（縮減資料至 2 ~ 3 個元素）**：如果 1000 個數字會出錯，那麼通常只需要 $N=2$ 或 $N=3$ 的迷你測資，就足以引爆完全相同的臭蟲！例如 `[1, 2]`、`[2, 1]`、或是 `[0]`。
2. **剔除所有干擾變因**：不要依賴外部複雜的讀檔或大迴圈，直接在代碼中宣告固定的小串列（如 `test_arr = [3, 1]`），專注測試最核心的那一小段計算邏輯。
3. **可重現性（100% 必然觸發）**：只要輸入這個微型測資，錯誤必然每次穩定重現，讓你能隨心所欲地進行單步追蹤與驗證。

記住：**「把問題縮減到一張便利貼寫得下的大小，臭蟲就已經解決了一半！」**


In [ ]:
# 13.8.1 程式碼演示：從龐大測資提煉極小可重現案例（MRE）
# 情境：某個尋找區間最大差值的函式在百萬筆測資時算錯

def buggy_max_profit(prices):
    # 潛在臭蟲：內部初值設定錯誤或索引偏差
    min_price = prices[0]
    max_diff = 0
    # 錯誤：若只考慮價格遞增，遇到全跌情境會怎樣？
    for p in prices:
        if p < min_price:
            min_price = p
        diff = p - min_price
        if diff > max_diff:
            max_diff = diff
    return max_diff

print("--- 面對巨大測資（難以人腦追蹤）---")
huge_prices = [100, 95, 92, 88, 85, 80, 75, 70, 60, 50] # 持續下跌
print("巨大跌價測資輸出:", buggy_max_profit(huge_prices))

print("\n--- 提煉極小可重現案例 (MRE: 僅需 2 個元素) ---")
# 縮減為僅有兩天的下跌情境：[10, 5]
mini_prices = [10, 5]
print("MRE 測資 [10, 5] 輸出:", buggy_max_profit(mini_prices))
print("思考：若題目要求『即使賠錢也必須買賣一次並求最小虧損（負數差值）』，")
print("原函式回傳 0 顯然不合題意！透過 2 個數字立刻精準洞察邏輯漏洞！")


### 13.8.1 語法重點回顧與核心觀念提煉

提煉 MRE 的三大黃金步驟：
1. **砍尺寸**：測資從 $N=10^5$ 砍到 $N=3$（例如 `[1, 2, 3]` 或 `[3, 2, 1]`）。
2. **砍分支**：把無關緊要的輸入輸出修飾語、複雜格式化全部註解掉，只保留核心函式呼叫。
3. **紙上推演對照**：對這 3 個微型數字，自己拿筆在草稿紙上手算一遍正確答案，然後與電腦算出的結果進行逐一比對，立即鎖定邏輯偏離點。


In [ ]:
# 13.8.1 學生實作練習：反轉部分串列之微型測試重現
# 任務說明：實作 reverse_sublist(arr, start, end) 函式
# 題意要求：將 arr 串列中從索引 start 到 end（包含兩端點）的子區段進行反轉，其餘部分維持原樣
# 請實作正確的切片反轉並組裝新串列，並使用微型測資驗證無誤！

def reverse_sublist(arr: list, start: int, end: int) -> list:
    # 請在此處實作精準區間反轉
    prefix = arr[:start]
    mid = arr[start:end + 1][::-1]
    suffix = arr[end + 1:]
    return prefix + mid + suffix

# 測試用例
data = [10, 20, 30, 40, 50]
print("反轉 [1, 3] 區間 (20, 30, 40 -> 40, 30, 20):")
print("結果:", reverse_sublist(data, 1, 3))


In [ ]:
# 13.8.1 單元測試驗證
assert reverse_sublist([1, 2, 3, 4, 5], 1, 3) == [1, 4, 3, 2, 5]
assert reverse_sublist([1, 2], 0, 1) == [2, 1]
assert reverse_sublist([10], 0, 0) == [10]
assert reverse_sublist([1, 2, 3], 0, 2) == [3, 2, 1]
print("13.8.1 單元測試全數通過！")


### 13.8.2 故障隔離法（Divide and Conquer）：二分註解程式碼，10 秒內鎖定臭蟲行段

在考試時，有時程式碼執行到一半莫名拋出 RE（例如不知哪裡越界了），或是輸出算出的值很奇怪，但整份程式有將近 100 行，到底該如何「在 10 秒內精確揪出到底是哪一行出了問題」？

千萬不要從第 1 行一個字一個字肉眼掃描到第 100 行！
高手採用的除錯戰術是演算法中最經典的思維：**「故障隔離法（Fault Isolation / Divide and Conquer，二分註解法）」**！

操作流程極其直覺且強悍：
1. **折半切斷**：在整份程式的正中央（第 50 行），直接放下一句 `print("=== 抵達前半段檢查點 ==="); exit(0)`（或在 Colab 中直接加這行並提早結束）。
2. **觀察反應**：
   - **如果前半段就已經出錯或崩潰**：代表臭蟲「100% 存在於第 1 到 50 行之間」！後半段的 50 行完全是無辜的，你瞬間縮小了一半的排查範圍！
   - **如果前半段非常順暢且印出了檢查點**：代表前半段完全健康，臭蟲「100% 潛伏在第 51 到 100 行之間」！
3. **再次折半**：對鎖定的那半邊再次進行折半，只需重複 3 到 4 次（$\log_2 100 \approx 7$），你就能在 10 秒之內，將臭蟲精確封鎖在一兩行代碼的夾縫之中！

這就是分治法的暴力美學：**不憑猜測，用二分法讓臭蟲無所遁形**！


In [ ]:
# 13.8.2 程式碼演示：二分註解隔離法模擬演練
def simulate_complex_pipeline(data):
    print("--- 開始執行複雜數據處理管線 ---")
    
    # 階段 1: 數據讀取與格式清理 (行 1-10)
    cleaned = [x.strip() for x in data if x.strip()]
    
    # 階段 2: 數值轉換與過濾 (行 11-20)
    numbers = [int(x) for x in cleaned]
    
    # ---------------- 假設在此處設立二分檢查點 ----------------
    # print(f"[二分檢查點 A] numbers={numbers}"); return "HALF_CHECK_OK"
    # --------------------------------------------------------
    
    # 階段 3: 核心運算與指標聚合 (行 21-30) - 潛伏除以零臭蟲
    total = sum(numbers)
    # 假設某一項運算除以了特定統計值
    stat_val = len([x for x in numbers if x > 100]) # 若沒有大於 100 者，長度為 0！
    
    # ---------------- 假設在此處設立二分檢查點 ----------------
    # print(f"[二分檢查點 B] stat_val={stat_val}")
    # --------------------------------------------------------
    
    avg_high = total / stat_val # 若 stat_val 為 0 引爆 ZeroDivisionError！
    
    # 階段 4: 格式化輸出
    return f"Result: {avg_high}"

# 測試用例：輸入沒有大於 100 的數字
try:
    simulate_complex_pipeline(["10", "20", "30"])
except Exception as e:
    print(f"\n[崩潰回報] 捕捉到例外: {type(e).__name__}: {e}")
    print("隔離結論：透過在階段 2 與階段 3 之間加入 print 檢查，")
    print("能立刻確認 stage 1 & 2 正常，迅速鎖定 stage 3 的分母計算缺失！")


### 13.8.2 語法重點回顧與核心觀念提煉

二分隔離法的操作口訣：
1. **中點下樁**：在疑似出錯區塊的正中央印出當前關鍵變數的值。
2. **驗收分界**：
   - 樁點前正常 $\implies$ 往後半段切。
   - 樁點前異常 $\implies$ 往前半段切。
3. **即時還原**：一旦鎖定出錯行，立刻修復邏輯，並將二分樁點註解或刪除，恢復主程式流暢運作。


In [ ]:
# 13.8.2 學生實作練習：分階段安全管線修復
# 任務說明：實作 safe_pipeline_executor(items) 函式
# 處理字串串列 items：
# 階段 1：將字串轉為整數串列（若包含非數字略過）
# 階段 2：若轉換後的串列為空，安全回傳 0（隔離防禦除以零）
# 階段 3：計算整數串列的平均值並回傳（浮點數）

def safe_pipeline_executor(items: list) -> float:
    # 階段 1
    nums = []
    for s in items:
        if s.lstrip('-').isdigit():
            nums.append(int(s))
            
    # 階段 2: 隔離防禦
    if not nums:
        return 0.0
        
    # 階段 3
    return sum(nums) / len(nums)

# 測試用例
print("正常平均:", safe_pipeline_executor(["10", "20", "30"]))
print("空資料防禦:", safe_pipeline_executor(["abc", "xyz"]))


In [ ]:
# 13.8.2 單元測試驗證
assert safe_pipeline_executor(["10", "20", "30"]) == 20.0
assert safe_pipeline_executor(["abc"]) == 0.0
assert safe_pipeline_executor([]) == 0.0
assert safe_pipeline_executor(["5", "invalid", "15"]) == 10.0
print("13.8.2 單元測試全數通過！")


### 13.8.3 標籤化 Print 追蹤技巧：以 print(f"[DEBUG] ...") 透視迴圈狀態

許多初學者在除錯時，最常在代碼裡隨手寫滿零散的 `print(x)`、`print(111)`、`print("aaa")`。
當程式跑起來時，終端機瞬間被幾十行無意義的數字與英文字母淹沒。初學者看著一堆 `111`、`222`、`0`、`5`，根本不知道這個數字到底是在哪個迴圈印出來的、對應哪一個變數，反而讓場面陷入更大的混亂！

資深選手在考場上進行 Print 追蹤時，有一套嚴格的專業規範——**「標籤化結構列印（Tagged Print Tracing）」**！

標籤化列印的三大黃金原則：
1. **永遠加上明顯的標籤前綴 `[DEBUG]`**：
   在所有除錯用的 print 前面一律統一加上 `[DEBUG]` 或 `[TRACE]`。這樣做有雙重奇效：第一，在眼花撩亂的終端機輸出一眼就能識別；第二，在解完題目準備送出代碼前，可以在編輯器中直接**「全文搜尋 `[DEBUG]`」**，一秒不漏地全數刪除，防止 debug print 污染標準輸出而吃到 WA！
2. **具名輸出變數名與當前迴圈計數器**：
   絕不寫 `print(val)`，而是永遠寫：`print(f"[DEBUG] 迭代 i={i}, target={target}, cur_val={val}")`！
3. **加入觸發條件過濾（只印關鍵輪次）**：
   若迴圈跑 10 萬次，全部印出來會直接讓螢幕卡死；加入條件篩選：`if i < 5 or i == n - 1:`，只印前 5 輪與最後一輪！


In [ ]:
# 13.8.3 程式碼演示：混亂盲印 vs 專業標籤化條件追蹤
test_data = [12, 45, 68, 23, 89]

print("--- 壞習慣：隨手盲印（完全看不出意義）---")
for i, x in enumerate(test_data):
    if x > 50:
        print(x) # 終端機只看到 68, 89，不知第幾輪何處觸發

print("\n--- 好習慣：專業標籤化條件追蹤 [DEBUG] ---")
for i, x in enumerate(test_data):
    # 明確標註輪次、變數名稱與狀態
    if x > 50:
        print(f"[DEBUG] >> 命中門檻！輪次 i={i}, 數值 x={x}, 是否為偶數={x % 2 == 0}")
    else:
        # 僅在特定關鍵時刻印出
        pass

print("\n送出前只需搜尋 '[DEBUG]'，即可在 3 秒內乾淨清空所有除錯行！")


### 13.8.3 語法重點回顧與核心觀念提煉

標籤化列印模板大賞：
1. **迴圈狀態追蹤模板**：
   ```python
   print(f"[DEBUG] step={step} | left={left}, right={right}, mid={mid}")
   ```
2. **二維網格局部快照模板**：
   ```python
   print(f"[DEBUG] 走訪座標 ({r}, {c}) = {grid[r][c]}")
   ```
3. **考場收尾鐵律**：送出代碼前，務必利用編輯器的搜尋快捷鍵（Ctrl+F），鍵入 `[DEBUG]`，逐一註解或刪除！因為 Online Judge 會把你印出的 `[DEBUG]` 一字不差地當作答案拿去比對，殘留任何一行都會直接拿到 WA！


In [ ]:
# 13.8.3 學生實作練習：二分搜尋追蹤器
# 任務說明：實作 binary_search_with_logs(arr, target) 函式
# 在已排序串列 arr 中使用二分搜尋尋找 target
# 需求：
# 1. 每次計算 mid = (left + right) // 2 時，產生格式化日誌字串：
#    f"[DEBUG] L={left}, R={right}, M={mid}, val={arr[mid]}"
#    並將該字串依序存入 log_history 串列中
# 2. 若找到回傳 (mid, log_history)
# 3. 若找不到回傳 (-1, log_history)

def binary_search_with_logs(arr: list, target: int) -> tuple:
    left = 0
    right = len(arr) - 1
    log_history = []
    
    while left <= right:
        mid = (left + right) // 2
        log_history.append(f"[DEBUG] L={left}, R={right}, M={mid}, val={arr[mid]}")
        
        if arr[mid] == target:
            return (mid, log_history)
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
            
    return (-1, log_history)

# 測試用例
data = [10, 20, 30, 40, 50, 60, 70]
idx, logs = binary_search_with_logs(data, 50)
print(f"找到索引: {idx}")
print("搜尋日誌歷程:")
for line in logs:
    print(" ", line)


In [ ]:
# 13.8.3 單元測試驗證
pos, history = binary_search_with_logs([1, 3, 5, 7, 9], 7)
assert pos == 3
assert len(history) >= 1
assert history[0].startswith("[DEBUG]")
pos_nf, _ = binary_search_with_logs([1, 2, 3], 99)
assert pos_nf == -1
print("13.8.3 單元測試全數通過！")


### 13.8.4 斷言（assert）防禦機制：在關鍵計算處設下「不變量」自我檢查

在複雜演算法（如動態規劃、二分搜尋、圖論 BFS/DFS）的實作中，我們心中通常會對某些狀態抱持著理所當然的假設，例如：「跑完這一段後，串列長度必須剛好等於 $N$」、「指針 `left` 絕對不可能大於 `right + 1`」、「機率加總必須介於 0 與 1 之間」。

然而，當程式存在細微漏洞時，這些假設往往會被悄悄打破，而你卻渾然不知，直到十幾步之後整個邏輯大崩潰。

Python 提供了一個專為工程師除錯而生的原生關鍵字：**`assert`（斷言）**！
其語法非常簡短：
```python
assert 條件判斷式, "條件不成立時的自訂報錯文字"
```
它的機制是：
- 若「條件判斷式」為 `True`，直譯器當作什麼都沒發生，以最高效率放行通過。
- 一旦「條件判斷式」為 `False`，直譯器會**「立刻在當下這行引爆 `AssertionError`」**，並印出你的報錯文字！

在演算法中，這被稱為設下**「不變量守衛（Invariant Guard）」**。
在考場寫下複雜邏輯時，在關鍵結算處補上一行 `assert`，能讓潛伏的幽靈狀態在第一秒就被當場抓包，省下數十分鐘的盲猜除錯時間！


In [ ]:
# 13.8.4 程式碼演示：使用 assert 守護陣列機率分佈不變量
def process_probabilities(prob_list):
    print(f"\n[檢查機率串列] {prob_list}")
    
    # 斷言 1: 機率串列不可為空
    assert len(prob_list) > 0, "機率串列不可為空！"
    
    # 斷言 2: 每個單項機率必須落在 [0.0, 1.0] 區間內
    for p in prob_list:
        assert 0.0 <= p <= 1.0, f"單項機率數值非法: p={p}"
        
    total_prob = sum(prob_list)
    # 斷言 3: 總和必須非常接近 1.0 (容許浮點誤差)
    assert abs(total_prob - 1.0) < 1e-5, f"機率總和不為 1: sum={total_prob}"
    
    print(f"✅ 不變量檢查全數通過！總機率為 {total_prob:.2f}")
    return True

# 案例 A: 合法機率分佈
process_probabilities([0.3, 0.5, 0.2])

# 案例 B: 總和破表引爆 assert
try:
    process_probabilities([0.6, 0.7]) # 總和 1.3，不合法！
except AssertionError as e:
    print(f"❌ 斷言攔截成功！AssertionError: {e}")


### 13.8.4 語法重點回顧與核心觀念提煉

使用 assert 的三大考場智慧：
1. **抓假定、設斷言**：在任何「你覺得絕對不可能發生」的地方，大膽寫下 `assert`。
2. **雙指標不變量**：在二分搜尋結束時，可驗證 `assert left >= 0 and right <= len(arr)`。
3. **送出考量**：在 Python 執行時加上 `-O` 參數會自動忽略 assert，但在一般評判系統中，未刪除的 assert 若觸發會被判定為 RE。因此，assert 是你在本機研發與驗證邏輯時最強的自我檢驗工具！


In [ ]:
# 13.8.4 學生實作練習：安全除法與商餘不變量驗證
# 任務說明：實作 divide_and_assert_invariant(dividend, divisor) 函式
# 1. 斷言 divisor != 0，若為 0 拋出自訂錯誤 "Divisor cannot be zero"
# 2. 計算整數商 q = dividend // divisor 與餘數 r = dividend % divisor
# 3. 斷言歐幾里得除法不變量：assert divisor * q + r == dividend
# 4. 回傳 (q, r) 元組

def divide_and_assert_invariant(dividend: int, divisor: int) -> tuple:
    # 1. 斷言除數不為 0
    assert divisor != 0, "Divisor cannot be zero"
    
    # 2. 計算商與餘
    q = dividend // divisor
    r = dividend % divisor
    
    # 3. 斷言除法不變量
    assert divisor * q + r == dividend, "Math Invariant Failed"
    
    return (q, r)

# 測試用例
print("29 除以 6:", divide_and_assert_invariant(29, 6))
print("-15 除以 4:", divide_and_assert_invariant(-15, 4))


In [ ]:
# 13.8.4 單元測試驗證
assert divide_and_assert_invariant(10, 3) == (3, 1)
assert divide_and_assert_invariant(20, 5) == (4, 0)
try:
    divide_and_assert_invariant(10, 0)
    assert False, "應引爆 AssertionError"
except AssertionError:
    pass
print("13.8.4 單元測試全數通過！")


### 13.8.5 邊界測資自測清單（Corner Cases Checklist）：N=0, 1、極值、全相同與有序

在 APCS 考場上，許多題目會提供 2 到 3 組簡單的「範例測資」。許多考生看著自己的程式碼順利跑過這 3 組範例，便興高采烈地按下送出，結果卻只拿到 20 分或 40 分（只有部分子任務得分）！

為什麼？因為**題目給的範例測資往往是最典型、最溫和的中間狀態**！而評判系統後端藏著的幾十組「隱藏測試點」，全部都是專門針對演算法邊界死角設計的**「極端測資（Corner Cases / Edge Cases）」**！

考場必備五大極端邊界檢查清單：
1. **資料規模極小值**：
   - $N=0$（空陣列、空字串）：程式是否會崩潰（IndexError）？
   - $N=1$（只有一個數字）：需要比對相鄰兩數或求次大值時，是否會越界？
2. **數值極值與負數**：
   - 全為負數（如 `[-5, -12, -3]`）：找最大值若初始值誤設為 `max_val = 0`，答案就會算出錯的 0！
   - 含有零（0）或除以零的邊界。
3. **全相同數值（All Identical）**：
   - 例如 `[7, 7, 7, 7]`：排序、去重、或找相鄰差值時，是否會陷入死迴圈或誤判？
4. **極端排列順序**：
   - 已經嚴格遞增升序（已排序）。
   - 嚴格遞減逆序（最差排列）。
5. **重複元素與無解情況**：
   - 查無符合條件的答案時，是否能依照題目要求正確印出 `-1` 或 `None`？

在點擊送出前，花 30 秒用這五大情境在心中快速掃描一遍，就能保住至少 30% 以上的隱藏分數！


In [ ]:
# 13.8.5 程式碼演示：全負數數列找最大子陣列和之初始值死角
# 題目需求：找出連續子陣列的最大總和 (Kadane's Algorithm 原型)

def buggy_max_subarray(nums):
    # 致命初值設定：誤以為最大總和至少是 0
    max_so_far = 0 
    current_sum = 0
    for x in nums:
        current_sum = max(x, current_sum + x)
        max_so_far = max(max_so_far, current_sum)
    return max_so_far

def correct_max_subarray(nums):
    # 正確初值防禦：以第一個元素作為初始底限，或 float('-inf')
    if not nums:
        return 0
    max_so_far = nums[0]
    current_sum = nums[0]
    for x in nums[1:]:
        current_sum = max(x, current_sum + x)
        max_so_far = max(max_so_far, current_sum)
    return max_so_far

# 一般常規測資（有正有負，兩者皆能算出正確答案 7）
normal_data = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
print("常規測資 [有正有負] 測試:")
print("  buggy 輸出:", buggy_max_subarray(normal_data))
print("  correct 輸出:", correct_max_subarray(normal_data))

# 極端測資：全負數！[-5, -2, -8]
# 正確答案應該是挑選最大負數 -2！
corner_data = [-5, -2, -8]
print("\n極端測資 [全為負數] 測試:")
print("  buggy 輸出:", buggy_max_subarray(corner_data), "❌ (回傳了 0，完全錯誤！)")
print("  correct 輸出:", correct_max_subarray(corner_data), "✅ (正確回傳最大單項 -2)")


### 13.8.5 語法重點回顧與核心觀念提煉

極值防禦黃金法則：
1. **極值初值初始化**：
   - 找最大值：初始值設為 `float('-inf')` 或 `nums[0]`，絕對不要隨便設 `0`！
   - 找最小值：初始值設為 `float('inf')` 或 `nums[0]`，絕對不要隨便設 `999999`（考場測資可能高達 $10^{18}$）！
2. **單一元素特判**：在函式開頭加入衛語句：`if len(arr) == 1: return arr[0]`。
3. **自備 5 組邊界小測資**：解完題後，隨手在下方寫 5 組測試呼叫，覆蓋空串列、單元素、全負數與全相同，無往不利！


In [ ]:
# 13.8.5 學生實作練習：健全最大值與次大值查找器
# 任務說明：實作 find_top_two(nums) 函式
# 給定整數串列 nums，找出前兩大不重複的數值，以元組 (first, second) 回傳（first > second）
# 嚴格邊界測資防禦要求：
# 1. 若不重複數值少於 2 個（如 nums 只有 1 個元素或全部數字皆相同），無法找出次大值，回傳 None！
# 2. 必須支援全為負數的情境（如 [-10, -5, -20] 正確回傳 (-5, -10)）！

def find_top_two(nums: list):
    # 請在此處進行健全邊界防禦與極值查找
    unique_sorted = sorted(list(set(nums)), reverse=True)
    if len(unique_sorted) < 2:
        return None
    return (unique_sorted[0], unique_sorted[1])

# 測試用例
print("常規測試:", find_top_two([10, 5, 20, 15]))
print("全負數測試:", find_top_two([-5, -10, -3]))
print("全相同邊界:", find_top_two([8, 8, 8]))
print("單元素邊界:", find_top_two([42]))


In [ ]:
# 13.8.5 單元測試驗證
assert find_top_two([1, 2, 3, 4]) == (4, 3)
assert find_top_two([-5, -1, -10]) == (-1, -5)
assert find_top_two([7, 7, 7]) is None
assert find_top_two([99]) is None
assert find_top_two([]) is None
print("13.8.5 單元測試全數通過！")


### 13.8.6 考場送出前 1 分鐘檢查 SOP：清空 debug print、檢查型態、範例再確認

在 APCS 考場的最後關頭，當你的解題思路已經完成、本機測試也看似正常時，**千萬不要在寫完最後一行的瞬間就急著按下送出（Submit）鍵**！
許多考生往往就是因為這最後 60 秒的粗心大意，讓原本能拿 100 分的程式碼白白掉了幾十分。

請將以下**「考場送出前 1 分鐘黃金檢查 SOP」**牢牢烙印在腦海中：

#### 步驟 1：地毯式搜尋並清除所有 `[DEBUG]` 列印（耗時 15 秒）
- 按下 Ctrl+F 搜尋 `print` 或 `DEBUG`。
- 確認所有中途加入的除錯印出語句已經全部刪除或加上註解 `#`。
- 確保留下的 `print()` 「只有且僅有題目要求輸出的最終結果」！

#### 步驟 2：輸入讀取純潔度檢查（耗時 10 秒）
- 檢查所有的 `input()` 是否為乾淨的空括號。
- 絕不可殘留任何 `input("請輸入：")` 提示字元。

#### 步驟 3：輸出型態與格式對齊確認（耗時 15 秒）
- 題目要求整數？確認是否誤印出浮點數（如 `4.0`）。
- 題目要求格式化小數？確認 f-string 位數是否吻合（如 `f"{x:.2f}"`）。
- 題目要求大小寫？再次比對範例輸出是 `YES` 還是 `Yes`。

#### 步驟 4：重新複製官方範例測資完整跑一次（耗時 20 秒）
- 將題目的範例輸入再次貼入執行，用肉眼仔細比對輸出的每一個字元、空格與換行是否與範例輸出 100% 嚴格吻合。

完成這四部曲，你的程式碼就具備了最頂級的抗錯與奪分能力，可以帶著無比的自信按下送出！


In [ ]:
# 13.8.6 程式碼演示：考場送出前 1 分鐘自我檢查清單驗證程式
def pre_submission_audit(code_str: str) -> list:
    warnings = []
    lines = code_str.split("\n")
    
    for i, line in enumerate(lines, 1):
        clean = line.strip()
        # 1. 檢查是否殘留 [DEBUG] 或 debug 列印
        if "[DEBUG]" in clean or "[debug]" in clean:
            warnings.append(f"行 {i}: ⚠️ 發現殘留的 debug 列印標籤！")
        # 2. 檢查 input() 是否含有引號提示詞
        if "input(" in clean and ('"' in clean or "'" in clean):
            # 排除純註解
            if not clean.startswith("#"):
                warnings.append(f"行 {i}: ❌ 發現 input() 帶有提示文字，會嚴重污染標準輸出！")
        # 3. 檢查浮點除法是否誤用於求商
        if " / " in clean and not ("//" in clean):
            warnings.append(f"行 {i}: ℹ️ 提示：發現單斜線浮點除法 '/'，請確認題目是否允許小數點。")
            
    return warnings

# 模擬一份準備送出的學生代碼
sample_submission = '''
import sys

# 讀取輸入
n = int(input("請輸入陣列長度："))
arr = list(map(int, input().split()))

# 核心計算
print(f"[DEBUG] 目前讀取到的陣列為: {arr}")
ans = sum(arr) / len(arr)

# 最終輸出
print(ans)
'''

print("=== 執行送出前程式碼自動稽核檢測 ===")
detected_issues = pre_submission_audit(sample_submission)
if not detected_issues:
    print("✅ 完美無瑕！完全符合考場送出規格，請安心送出！")
else:
    print(f"🚨 稽核系統攔截到 {len(detected_issues)} 處高危險潛在失分點：")
    for warn in detected_issues:
        print(" ", warn)


### 13.8.6 語法重點回顧與核心觀念提煉

考場心理與戰略收尾心法：
1. **冷靜是最高美德**：哪怕時間只剩下最後 5 分鐘，嚴格執行 1 分鐘 SOP 也比慌亂送出盲改代碼有效十倍。
2. **保分思維（Partial Scoring）**：若第三、四題演算法想不出最優解，先用前面學會的防禦架構寫出無 Bug 的暴力解（拿滿子任務的 40% 分數），積少成多才是 APCS 奪取 4 級分與 5 級分的制勝密碼！


In [ ]:
# 13.8.6 學生實作練習：考場全方位標準送出模板實戰
# 任務說明：實作 apcs_standard_solution(input_text) 函式
# 題目規格：
# 1. 輸入兩行：第一行整數 n，第二行 n 個整數
# 2. 題目要求：找出這 n 個整數中的最大值，以及能被 3 整除的數字個數
# 3. 輸出格式規格：
#    第一行輸出最大整數
#    第二行輸出能被 3 整除的個數
# 4. 嚴格要求：全流程乾淨純粹，無提示字元、無 debug print，格式完全精確對齊！

import io, sys

def apcs_standard_solution(input_text: str) -> str:
    old_stdin = sys.stdin
    old_stdout = sys.stdout
    sys.stdin = io.StringIO(input_text)
    sys.stdout = io.StringIO()
    
    # --- 考場標準解題架構 ---
    n = int(input())
    arr = list(map(int, input().split()))
    
    max_val = max(arr)
    count_div3 = sum(1 for x in arr if x % 3 == 0)
    
    print(max_val)
    print(count_div3)
    # -----------------------
    
    result_out = sys.stdout.getvalue()
    sys.stdin = old_stdin
    sys.stdout = old_stdout
    return result_out

# 測試用例
sample_in = "5\n10 15 20 33 7\n"
print("標準輸出結果:\n" + apcs_standard_solution(sample_in))


In [ ]:
# 13.8.6 單元測試驗證
test_in = "4\n3 6 9 12\n"
expected = "12\n4\n"
assert apcs_standard_solution(test_in) == expected

test_in_neg = "3\n-9 -3 -6\n"
expected_neg = "-3\n3\n"
assert apcs_standard_solution(test_in_neg) == expected_neg
print("13.8.6 單元測試全數通過！")


## 13.8 總結與全章總結：穿上防彈衣，迎戰 APCS 真題！

至此，恭喜你完整修畢了**第十三章《程式除錯（Debug）與異常處理》的全部 8 大核心單元**！
我們從 Online Judge 評判底層機制出發，經歷了語法排查、執行崩潰防禦、邏輯盲點掃除、效能坑洞填補、格式字元級對齊，最終在第 13-8 節總結出考場系統化除錯的黃金 SOP：

```mermaid
flowchart TD
    A["拿到題目"] --> B["看 N 定複雜度 (10^7 法則)"]
    B --> C["實作核心演算法"]
    C --> D{"本機執行測試"}
    D -- "噴紅字 CE/RE" --> E["閱讀 Traceback 行號與指標 / 二分隔離"]
    D -- "卡住跑很久" --> F["檢查 while 收斂與 list in / 改用 set"]
    D -- "結果算錯 WA" --> G["提取 3 元素 MRE / 檢查優先級與差一"]
    E --> D
    F --> D
    G --> D
    D -- "答案正確" --> H["帶入 5 大 Corner Cases (N=0,1/負數/全相同)"]
    H --> I["送出前 1 分鐘 SOP (清 debug / 查 input / 對格式)"]
    I --> J["🚀 點擊送出，全數綠燈 Accepted (AC)！"]
```

### 🏆 第十三章 8 節全景知識回顧
1. **13.1 評判體系**：stdin/stdout 導向，AC、CE、WA、TLE、RE、MLE 評判全解析。
2. **13.2 語法排查**：編譯期檢查機制，Traceback 行號游標，漏冒號/括號/縮排五大雷區。
3. **13.3 崩潰防禦**：IndexError、ValueError、KeyError、ZeroDivisionError 與 LBYL 護城河。
4. **13.4 例外捕捉**：try-except 原生防禦，嚴禁裸露 except，`EOFError` 讀取未知行數測資。
5. **13.5 邏輯掃除**：差一錯誤、括號優先級、浮點數精準度、二維串列淺拷貝幽靈、變數遮蔽。
6. **13.6 效能診斷**：1 秒 $10^7$ 次操作量級，死迴圈收斂，`set in`、`join()` 與極速 I/O。
7. **13.7 格式對齊**：字元級 diff 比對，杜絕 input 提示詞，`print(*ans)` 消滅行末空格，大小寫對齊。
8. **13.8 考場戰略**：極小重現案例（MRE）、二分隔離法、標籤化 print、assert 守衛與送出前 SOP。

---

### 🌟 邁向巔峰：進入第十四章《APCS 實作真題循序漸進解題特訓》！
你現在已經不是只會盲目寫代碼的初學者，而是一位裝備了完整自我診斷、錯誤隔離與考場防彈衣的準競賽選手！
在接下來的**第十四章**中，我們將正式跨入實戰最高殿堂——**迎戰連續 19 題真實歷屆 APCS 檢定實作真題（從 c294 三角形辨別到 g596 動線安排）**！帶著第 1 到 13 章扎下的雄厚內功，讓我們在真題考場上一展身手，奪下 APCS 實作題滿級分！
